# P94 — Stan: un lenguaje de programación probabilística

## 1. Título y paper

**Paper:** *Stan: A Probabilistic Programming Language*  
**Autoría:** Bob Carpenter, Andrew Gelman, Matthew D. Hoffman, Daniel Lee, Ben Goodrich, Michael Betancourt, Marcus Brubaker, Jiqiang Guo, Peter Li, Allen Riddell  
**Año y venue:** 2017 · Journal of Statistical Software, 76(1)  
**Nivel:** L3 · **Motor:** `programacion_probabilistica`  
**Ficha completa:** [`P94_programacion_probabilistica`](../../papers/foundational/P94_programacion_probabilistica/README.md)

**Hito:** Separa declarar el modelo de calcular la inferencia: se escribe qué se supone del mundo y el motor devuelve la posterior.

- [doi:10.18637/jss.v076.i01](https://doi.org/10.18637/jss.v076.i01)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Cada modelo bayesiano nuevo exigía escribir a mano su propio muestreador, con la matemática y los errores que eso trae. El coste de probar una variante del modelo era el de reimplementar el algoritmo.
2. Ejecutar una implementación mínima de la propuesta: Un lenguaje declarativo para especificar el modelo —previas y verosimilitud— y un motor de inferencia general basado en Monte Carlo hamiltoniano con NUTS, más diagnósticos de convergencia integrados.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P91
- Lunn et al. (2000), BUGS
- Neal (2011), HMC


## 4. Intuición

Escribes lo que supones del mundo —una previa y una verosimilitud— y el motor devuelve la posterior. No escribes el muestreador. Cambiar el modelo es cambiar dos líneas, no reimplementar el algoritmo, y ese cambio de coste es lo que hace posible probar variantes.


## 5. Concepto mínimo

```text
Modelo declarado:
    theta ~ Beta(2, 2)
    y[i]  ~ Bernoulli(theta)

El motor se encarga de:
    muestrear la posterior · diagnosticar convergencia · reportar incertidumbre

Separar QUÉ se supone de CÓMO se calcula.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('programacion_probabilistica', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué devuelve el motor: un número o una distribución?
2. ¿Coincide con el posterior analítico?
3. ¿Qué añade frente a la estimación puntual?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('programacion_probabilistica', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('programacion_probabilistica', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Devuelve la **posterior completa**: media 0,7936 e intervalo creíble del 90 % en [0,6844, 0,8835]. Coincide con el posterior analítico Beta(35, 9), de media 0,7955 — el motor no inventa, resuelve lo que el modelo implica. Y la estimación puntual de máxima verosimilitud (0,825) no dice **nada** sobre la incertidumbre.


## 10. Comentario pedagógico

Con 40 lanzamientos ese intervalo es ancho, y el ancho **es** el resultado: informa de cuánto no sabemos. Reportar solo la media de una posterior desperdicia justamente lo que distingue a este enfoque, y es lo que ocurre en la mayoría de los usos que se ven por ahí.


## 11. Error o anti-patrón deliberado

Anti-patrón: creerse una posterior sin mirar los diagnósticos de convergencia.


In [ ]:
print('Una cadena que no ha convergido produce una posterior con aspecto perfecto.')
print('Antes de leer cualquier numero: R-hat, tamano efectivo de muestra y divergencias.')
print('Stan los reporta por defecto justamente porque nadie los miraria si no.')

## 12. Corrección

Lo que el motor sí garantiza, comprobado contra la solución exacta:


In [ ]:
r = run_paper_lab('programacion_probabilistica', seed=7)['result']
print('modelo declarado :', r['modelo_declarado'])
print('posterior muestreada:', r['posterior_por_muestreo'])
print('posterior analitico :', r['posterior_analitico_beta'])
print('coinciden:', r['coincide_muestreo_con_analitico'])

## 13. Desafío guiado

Cambia mentalmente la previa a Beta(20, 2) —una creencia fuerte en que la moneda está sesgada hacia cara— y predice hacia dónde se moverá la posterior con los mismos 40 datos.


In [ ]:
r = run_paper_lab('programacion_probabilistica', seed=3)['result']
show(r)

## 14. Desafío autónomo

Escribe un modelo bayesiano de un problema tuyo en un lenguaje probabilístico real, con previas justificadas. Reporta la posterior completa y los diagnósticos, no solo la media.


## 15. Evidencia de aprendizaje

Guarda el modelo declarado, la posterior con su intervalo y tu comparación con la estimación puntual.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P94_programacion_probabilistica/README.md) · evaluación formal: [`assessments/papers/P94_programacion_probabilistica.md`](../../assessments/papers/P94_programacion_probabilistica.md)


## 16. Cierre

Ya se puede declarar un modelo y obtener incertidumbre honesta. Queda la pregunta que ningún modelo de asociación responde: qué pasa si intervengo.


## 17. Conexión con el siguiente hito

- P95

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
